<a href="https://colab.research.google.com/github/TeeMiles01/Data-Science-projects/blob/main/wikipedia-scraper/webscraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
def get_html_content(url):
    """Fetch a Wikipedia page and return a parsed BeautifulSoup object."""
    headers = {
        "User-Agent": "MyWikipediaScraper/1.0 (chrispette@gmail.com)"
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    return soup

In [ ]:
def get_title(soup):
    """Extract the article's main title."""
    title_tag = soup.find("h1", {"id": "firstHeading"})
    return title_tag.get_text(strip=True) if title_tag else None

In [ ]:
def get_text_by_heading(soup):
    """Extract paragraph text grouped under their respective headings."""
    content_div = soup.find("div", {"id": "mw-content-text"})

    content_dict = {}
    current_heading = "Introduction"  # text before the first heading
    content_dict[current_heading] = []

    # Only look inside the main parser output to avoid sidebars/infoboxes noise
    parser_output = content_div.find("div", {"class": "mw-parser-output"})

    for element in parser_output.find_all(["h2", "h3", "h4", "p"], recursive=True):
        if element.name in ["h2", "h3", "h4"]:
            # Wikipedia headings often contain a "span.mw-headline" or similar
            heading_text = element.get_text(strip=True)
            # Remove trailing "[edit]" links some Wikipedia versions include
            heading_text = heading_text.replace("[edit]", "").strip()
            if heading_text:
                current_heading = heading_text
                content_dict[current_heading] = []
        elif element.name == "p":
            paragraph_text = element.get_text(strip=True)
            if paragraph_text:  # skip empty paragraphs
                content_dict[current_heading].append(paragraph_text)

    return content_dict

In [ ]:
def get_internal_links(soup):
    """Collect all internal Wikipedia article links (as full URLs)."""
    content_div = soup.find("div", {"id": "mw-content-text"})
    links = set()  # use a set to avoid duplicates

    for a_tag in content_div.find_all("a", href=True):
        href = a_tag["href"]
        if href.startswith("/wiki/") and ":" not in href.split("/wiki/")[1]:
            full_url = "https://en.wikipedia.org" + href
            links.add(full_url)

    return list(links)

In [ ]:
def scrape_wikipedia_page(url):
    """
    Given a Wikipedia URL, return a dictionary with:
    - title
    - content (headings mapped to paragraphs)
    - internal_links
    """
    soup = get_html_content(url)

    result = {
        "title": get_title(soup),
        "content": get_text_by_heading(soup),
        "internal_links": get_internal_links(soup)
    }

    return result

In [ ]:
url = "https://en.wikipedia.org/wiki/Data_Science"
data = scrape_wikipedia_page(url)

print("Title:", data["title"])
print("\nNumber of sections:", len(data["content"]))
print("\nFirst 300 characters of Introduction:")
print(data["content"]["Introduction"][0][:300])

print("\nTotal internal links found:", len(data["internal_links"]))
print("Sample links:", data["internal_links"][:5])

Title: Data science

Number of sections: 11

First 300 characters of Introduction:
Data scienceis aninterdisciplinaryacademic field[1]that usesstatistics,scientific computing,scientific methods, processing,scientific visualization,algorithms, coding (like Python, SQL, and R), and systems to extract or extrapolateknowledgefrom potentially noisy,structured, orunstructured data.[2]Ad

Total internal links found: 0
Sample links: []
